# 🛡️ Level 2 - Project 3: Fraud Detection Pipeline

**Domain:** Anomaly Detection & Binary Classification  
**Internship Track:** Data Analytics Internship (Oasis Infobyte)  
**Author:** Akshay Anil Kumar  

---

## 1. Automated Extraction & Class Imbalance Audit
This stage automatically scans the workspace for the project archive, extracts the data tables, and inspects the distribution ratio between legitimate transactions and fraudulent activity.

In [1]:
import zipfile
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# 1. Find all zip files in the current folder
zip_files = glob.glob("*.zip")
print(f"🔍 Found {len(zip_files)} zip file(s) in folder.")

fraud_zip = None
for z in zip_files:
    lower_name = z.lower()
    if any(keyword in lower_name for keyword in ["fraud", "credit", "card", "transaction", "archive"]):
        with zipfile.ZipFile(z, 'r') as zip_ref:
            members = zip_ref.namelist()
            print(f"\n📦 Archive: {z}")
            for m in members:
                print(f"  - {m}")
            if any(any(keyword in m.lower() for keyword in ["fraud", "credit", "card", "transaction"]) for m in members):
                fraud_zip = z
                break

if fraud_zip is None and zip_files:
    # Fall back to the first zip if no strong keyword match was found
    fraud_zip = zip_files[0]
    print(f"\n⚠️ No obvious fraud archive name found; falling back to first zip: {fraud_zip}")

if fraud_zip:
    print(f"\n✅ Extracting archive: {fraud_zip}")
    with zipfile.ZipFile(fraud_zip, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✓ Extraction complete!")
else:
    print("⚠️ No zip archive found to extract.")

# 2. Locate candidate CSVs after extraction
csv_candidates = (
    glob.glob("*fraud*.csv") +
    glob.glob("*credit*.csv") +
    glob.glob("*card*.csv") +
    glob.glob("*transaction*.csv") +
    glob.glob("*.csv")
)
csv_candidates = [c for c in csv_candidates if all(exclude not in c.lower() for exclude in ["housing", "wine", "menu", "retail"])]

print(f"\n🔎 Found {len(csv_candidates)} candidate CSV file(s).")
for idx, candidate in enumerate(csv_candidates, 1):
    print(f"  {idx}. {candidate}")

if not csv_candidates:
    print("⚠️ No CSV candidates found — generating a synthetic fraud dataset for demonstration.")
    np.random.seed(42)
    n = 5000
    df = pd.DataFrame({
        'transaction_amt': np.random.exponential(scale=50, size=n),
        'oldbalanceOrg': np.random.uniform(0, 10000, size=n),
        'newbalanceOrig': np.random.uniform(0, 10000, size=n),
        'type': np.random.choice(['PAYMENT','TRANSFER','CASH_OUT','DEBIT'], size=n),
        'is_fraud': np.random.choice([0,1], size=n, p=[0.98,0.02])
    })
    print(f"✓ Synthetic data created. Shape: {df.shape}")
else:
    # 3. Load the first likely fraud dataset
    csv_files = [c for c in csv_candidates if any(keyword in c.lower() for keyword in ["fraud", "credit", "card", "transaction"])]
    if not csv_files:
        csv_files = csv_candidates

    csv_file = csv_files[0]
    print(f"\n📊 Loading dataset from: '{csv_file}'")
    try:
        df = pd.read_csv(csv_file)
        print(f"✓ Data successfully loaded. Shape: {df.shape}")
    except Exception as e:
        print(f"⚠️ Failed to read '{csv_file}': {e}\nGenerating synthetic fallback dataset.")
        np.random.seed(42)
        n = 5000
        df = pd.DataFrame({
            'transaction_amt': np.random.exponential(scale=50, size=n),
            'oldbalanceOrg': np.random.uniform(0, 10000, size=n),
            'newbalanceOrig': np.random.uniform(0, 10000, size=n),
            'type': np.random.choice(['PAYMENT','TRANSFER','CASH_OUT','DEBIT'], size=n),
            'is_fraud': np.random.choice([0,1], size=n, p=[0.98,0.02])
        })
        print(f"✓ Synthetic data created. Shape: {df.shape}")

# 4. Identify the target column and preview the data
target_cols = [col for col in df.columns if col.lower() in ['class', 'is_fraud', 'fraud', 'target']]
if not target_cols:
    raise ValueError(f"❌ No fraud target column found in CSV columns: {list(df.columns)}")

target_col = target_cols[0]
print(f"Target column: {target_col}")
print("\n=== Class Distribution (0 = Normal, 1 = Fraud) ===")
print(df[target_col].value_counts())
print("\n=== Dataset Structure Preview ===")
display(df.head())

🔍 Found 0 zip file(s) in folder.
⚠️ No zip archive found to extract.

🔎 Found 0 candidate CSV file(s).
⚠️ No CSV candidates found — generating a synthetic fraud dataset for demonstration.
✓ Synthetic data created. Shape: (5000, 5)
Target column: is_fraud

=== Class Distribution (0 = Normal, 1 = Fraud) ===
is_fraud
0    4882
1     118
Name: count, dtype: int64

=== Dataset Structure Preview ===


,transaction_amt,oldbalanceOrg,newbalanceOrig,type,is_fraud
0,23.463404,3936.355203,3736.408185,DEBIT,0
1,150.506072,4734.356594,3329.120962,TRANSFER,0
2,65.837285,8545.473932,1761.539125,PAYMENT,0
3,45.647128,3400.043861,6072.666701,DEBIT,0
4,8.481244,8696.496848,4766.241605,PAYMENT,0


## 2. Feature Isolation & Partitioning
We split our independent variables from the target tracking class and partition the structural metrics into an **80% training matrix** and a **20% testing split** using stratified matching to preserve the fraud ratio.

In [ ]:
# Separate features and target
X = df.drop([target_col], axis=1)
y = df[target_col]

# Preprocess features: encode categoricals and ensure numeric matrix for the classifier
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Fill missing values and keep numeric columns only
X = X.select_dtypes(include=[np.number]).fillna(0)

# Use stratify=y to make sure both training and testing sets get an equal percentage of fraud cases
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features size: {X_train.shape}")
print(f"Testing features size: {X_test.shape}")

NameError: name 'df' is not defined

## 3. Classifier Model Training & Performance Verification
We train an ensemble Random Forest model to flag highly suspicious structural parameters and map the classification errors using a precision-focused confusion matrix.

In [3]:
# Initialize and fit the model
fraud_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
fraud_model.fit(X_train, y_train)

# Run test predictions
y_pred = fraud_model.predict(X_test)

print("=== Performance Evaluation ===")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Area Under ROC Curve (ROC-AUC): {roc_auc_score(y_test, y_pred):.4f}\n")
print("=== Detailed Classification Summary ===")
print(classification_report(y_test, y_pred))

# Render Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Reds', 
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
plt.title('Fraud Detection Performance Confusion Matrix', fontsize=12, pad=10)
plt.xlabel('Model Predicted Classification')
plt.ylabel('Actual Validation Label')
plt.show()

NameError: name 'X_train' is not defined